In [80]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [81]:
df = pd.read_csv('insurance_data_v2.csv')
df

,age,affordability,have_insurance
0,56,5,1
1,69,8,1
2,46,10,1
3,32,9,1
4,60,9,1
...,...,...,...
95,57,7,1
96,38,2,1
97,33,3,1
98,62,1,0


In [82]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df[['age', 'affordability']], df.have_insurance, test_size=0.2, random_state=42)

In [83]:
X_train_scaled = X_train.copy()
X_train_scaled['age'] = X_train_scaled['age'] /100
X_train_scaled['affordability'] = X_train_scaled['affordability'] /10

X_test_scaled = X_test.copy()
X_test_scaled['age'] = X_test_scaled['age'] /100
X_test_scaled['affordability'] = X_test_scaled['affordability'] /10 

In [84]:
X_test_scaled

,age,affordability
83,0.61,0.1
53,0.42,0.4
70,0.53,0.7
45,0.54,0.7
44,0.20,0.7
39,0.68,0.5
22,0.61,1.0
80,0.46,0.8
10,0.28,0.1
0,0.56,0.5


In [85]:
model = keras.Sequential([
    keras.layers.Dense(
        1,
        input_shape=(2,),
        activation='sigmoid',
        kernel_initializer='ones',
        bias_initializer='zeros'
    )
])
model.compile(optimizer = 'adam',
              loss = 'binary_crossentropy',
              metrics = ['accuracy'])

model.fit(X_train_scaled,y_train, epochs=500)

c:\Users\boogey\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.8375 - loss: 0.4146
Epoch 2/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8375 - loss: 0.4139
Epoch 3/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8375 - loss: 0.4132
Epoch 4/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8375 - loss: 0.4124
Epoch 5/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8375 - loss: 0.4118
Epoch 6/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8375 - loss: 0.4111
Epoch 7/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8375 - loss: 0.4104
Epoch 8/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8375 - loss: 0.4098
Epoch 9/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8375 - loss: 0.4091
Epoch 10/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8375 - loss: 0.4084
Epoch 11/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8375 - loss: 0.4077
Epoch 12/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8375 - lo

In [86]:
model.evaluate(X_test_scaled, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 303ms/step - accuracy: 0.9000 - loss: 0.2751


[0.2751326858997345, 0.8999999761581421]

In [87]:
coef, intercept = model.get_weights()
coef, intercept

(array([[1.4153278],
        [1.9958687]], dtype=float32),
 array([0.10222723], dtype=float32))

In [88]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def prediction_function(age, affordability):
    age_scaled = age / 100.0
    affordability_scaled = affordability / 10.0

    weighted_sum = coef[0, 0] * age_scaled + coef[1, 0] * affordability_scaled + intercept[0]
    return sigmoid(weighted_sum)

In [89]:
prediction_function(42,4)

np.float32(0.81682944)

In [90]:
def log_loss(y_true, y_predicted):
    epsilon = 1e-15
    y_predicted_new = [max(i,epsilon) for i in y_predicted]
    y_predicted_new = [min(i,1-epsilon) for i in y_predicted_new]
    y_predicted_new = np.array(y_predicted_new)
    return -np.mean(y_true*np.log(y_predicted_new) + (1-y_true)*np.log(1-y_predicted_new))

In [91]:
def sigmoid_numpy(X):
    return 1/(1+np.exp(-X))
sigmoid_numpy(np.array([12,0,1]))

array([0.99999386, 0.5       , 0.73105858])

In [92]:
class myNM:
    def __init__(self):
        self.w1 = 1.0
        self.w2 = 1.0
        self.bias = 0.0

    def fit(self, X, y, epochs=500, rate=0.5):
        self.w1, self.w2, self.bias = self.gradient_descent(
            X['age'], X['affordability'], y, epochs, rate
        )
        return self

    def predict(self, X):
        age = np.asarray(X['age'], dtype=float)
        affordability = np.asarray(X['affordability'], dtype=float)
        weighted_sum = self.w1 * age + self.w2 * affordability + self.bias
        return sigmoid_numpy(weighted_sum)

    def gradient_descent(self, age, affordability, y_true, epochs, rate=0.5):
        age = np.asarray(age, dtype=float)
        affordability = np.asarray(affordability, dtype=float)
        y_true = np.asarray(y_true, dtype=float)

        w1 = 1.0
        w2 = 1.0
        bias = 0.0
        n = len(age)

        for i in range(epochs):
            weighted_sum = w1 * age + w2 * affordability + bias
            y_predicted = sigmoid_numpy(weighted_sum)

            loss = log_loss(y_true, y_predicted)

            w1d = (1 / n) * np.dot(age, (y_predicted - y_true))
            w2d = (1 / n) * np.dot(affordability, (y_predicted - y_true))
            bias_d = np.mean(y_predicted - y_true)

            w1 = w1 - rate * w1d
            w2 = w2 - rate * w2d
            bias = bias - rate * bias_d

            if i % 100 == 0 or i == epochs - 1:
                print(f'Epoch:{i}, w1:{w1}, w2:{w2}, bias:{bias}, loss:{loss}')

        return w1, w2, bias 

In [93]:
customModel = myNM()
customModel.fit(X_train_scaled, y_train, epochs=500, rate=0.5)

Epoch:0, w1:1.0273864280909306, w2:1.0510062327507723, bias:0.05354208459909282, loss:0.4148552018497753
Epoch:100, w1:1.3299628945821191, w2:2.6584265917687477, bias:-0.17307773310831337, loss:0.3206779376730634
Epoch:200, w1:1.5102516152324108, w2:3.578316067860308, bias:-0.6459394608584169, loss:0.2982852862128035
Epoch:300, w1:1.7203566806792105, w2:4.236668029218133, bias:-0.9928542733849757, loss:0.28619263123392663
Epoch:400, w1:1.9361629111867023, w2:4.729390856298151, bias:-1.265109882751809, loss:0.27886295161598573
Epoch:499, w1:2.145208055137999, w2:5.10785650263842, bias:-1.486248596262043, loss:0.27406842006515497


In [94]:
customModel.predict(X_test_scaled) 

array([0.58251686, 0.81120842, 0.96180882, 0.96258906, 0.92541429,
       0.92597341, 0.99282666, 0.97305997, 0.40738055, 0.90627656,
       0.46534769, 0.96763658, 0.97238294, 0.7290879 , 0.77614843,
       0.98784694, 0.99037691, 0.78781105, 0.97010091, 0.97071689])